# BENCHMARK SO SÁNH ĐA MÔ HÌNH HỆ THỐNG GỢI Ý VỚI ĐỒ THỊ TRI THỨC (CKAN)
### Đề tài: Knowledge Graph trong Hệ Thống Gợi Ý Điện Ảnh (Movie Recommendation System)
**Nhóm tác giả & Môi trường thực nghiệm**: Google Colab GPU (NVIDIA T4 / V100)

---
### Mục Tiêu Thực Nghiệm:
1. **So sánh đa phương pháp**:
   - **MostPopular**: Baseline không cá nhân hóa (gợi ý theo số đông).
   - **Item-KNN**: Lọc cộng tác truyền thống dựa trên láng giềng k-gần nhất.
   - **Matrix Factorization (Biased MF)**: Lọc cộng tác nhân tử hóa ma trận kinh điển (không dùng KG).
   - **CKAN (Collaborative Knowledge-aware Attentive Network)**: Mô hình học sâu đề xuất kết hợp Đồ thị tri thức (102,569 entities, 499,474 triples).
2. **Chứng minh tính ưu việt của Knowledge Graph**:
   - **Thực nghiệm CTR Prediction (Warm-start)**: So sánh AUC và F1.
   - **Thực nghiệm Data Sparsity (10%, 20%, 50%, 100% Data)**: Chứng minh MF bị sụp đổ khi tương tác thưa thớt, trong khi CKAN giữ vững độ chính xác nhờ tri thức thực thể.
   - **Thực nghiệm Cold-Start Simulation**: Chứng minh giải pháp gợi ý cho người dùng mới không có lịch sử.
   - **Thực nghiệm Top-K Ranking (Recall@K, Precision@K, NDCG@K)** trên toàn bộ kho phim.



In [ ]:
# ============================================================
# 1. KIỂM TRA PHẦN CỨNG & GPU
# ============================================================
import torch

print("=== KIỂM TRA PHẦN CỨNG GOOGLE COLAB ===")
print("Phiên bản PyTorch :", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị tính toán:", device)
if torch.cuda.is_available():
    print("Tên GPU           :", torch.cuda.get_device_name(0))
    print("Bộ nhớ GPU (VRAM) :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")



In [ ]:
# ============================================================
# 2. CÀI ĐẶT THƯ VIỆN BỔ TRỢ
# ============================================================
!pip install -q --upgrade scikit-learn matplotlib seaborn pandas tqdm

import os
import random
import collections
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

# Thiết lập seed tái lập kết quả
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("[OK] Đã tải xong thư viện và thiết lập Seed:", SEED)



In [ ]:
# ============================================================
# 3. THIẾT LẬP SIÊU THAM SỐ HUẤN LUYỆN
# ============================================================
DATASET = "movie"
DIM = 64              # Chiều nhúng Embedding
N_LAYER = 1           # Số bước lan truyền tri thức (L=1 tối ưu)
UTSS = 32             # User Triple Set Size
ITSS = 64             # Item Triple Set Size
AGG = "concat"        # Cơ chế gộp embedding
BATCH_SIZE = 2048     # Batch size huấn luyện
N_EPOCH = 20          # Số epochs
LEARNING_RATE = 0.002 # Tốc độ học Adam
L2_WEIGHT = 1e-5      # Trọng số điều chuẩn L2

DATA_DIR = f"./data/{DATASET}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./results", exist_ok=True)

print(f"Cấu hình: Dataset={DATASET}, Dim={DIM}, KG Layers={N_LAYER}, Batch={BATCH_SIZE}, Epochs={N_EPOCH}")



In [ ]:
# ============================================================
# 4. TẢI DỮ LIỆU ĐÃ TIỀN XỬ LÝ TỪ GITHUB
# ============================================================
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/Chinh-de/dss_ckan/main/backend/data/movie/"
FILES = ["ratings_final.npy", "kg_final.npy"]

for f in FILES:
    target = os.path.join(DATA_DIR, f)
    if not os.path.exists(target):
        url = BASE_URL + f
        print(f"Đang tải {f} từ {url}...")
        try:
            urllib.request.urlretrieve(url, target)
        except Exception as e:
            print(f"Lỗi tải trực tuyến: {e}. Vui lòng upload thủ công {f} vào thư mục {DATA_DIR}")

rating_np = np.load(os.path.join(DATA_DIR, "ratings_final.npy"))
kg_np = np.load(os.path.join(DATA_DIR, "kg_final.npy"))

n_user = int(len(set(rating_np[:, 0])))
n_item = int(len(set(rating_np[:, 1])))
n_entity = int(max(np.max(kg_np[:, 0]), np.max(kg_np[:, 2]))) + 1
n_relation = int(np.max(kg_np[:, 1])) + 1

print(f"Thống kê Dữ liệu:")
print(f"- Số người dùng (Users) : {n_user:,}")
print(f"- Số lượng phim (Items) : {n_item:,}")
print(f"- Tổng số tương tác     : {len(rating_np):,} (Cân bằng 50% Like - 50% Dislike)")
print(f"- Số thực thể KG        : {n_entity:,}")
print(f"- Số quan hệ KG         : {n_relation:,}")
print(f"- Tổng số bộ ba tri thức: {len(kg_np):,}")



In [ ]:
# ============================================================
# 5. CHIA TẬP DỮ LIỆU (6:2:2) & KHỞI TẠO LAN TRUYỀN TRI THỨC
# ============================================================
def dataset_split(rating_np):
    n_rows = rating_np.shape[0]
    indices = np.random.permutation(n_rows)
    train_idx = indices[:int(0.6 * n_rows)]
    eval_idx = indices[int(0.6 * n_rows):int(0.8 * n_rows)]
    test_idx = indices[int(0.8 * n_rows):]
    return rating_np[train_idx], rating_np[eval_idx], rating_np[test_idx]

train_data, eval_data, test_data = dataset_split(rating_np)
print(f"Phân chia tập: Train={len(train_data):,}, Eval={len(eval_data):,}, Test={len(test_data):,}")

# Xây dựng từ điển đồ thị tri thức
kg_dict = collections.defaultdict(list)
for head, relation, tail in kg_np:
    kg_dict[head].append((tail, relation))

# Khởi tạo ripple set cho items và users
print("Đang xây dựng Item Triple Set (ITSS=64)...")
item_triple_set = collections.defaultdict(list)
for item in range(n_item):
    for l in range(N_LAYER):
        h, r, t = [], [], []
        neighbors = kg_dict.get(item, [])
        if len(neighbors) == 0:
            h = [item] * ITSS
            r = [0] * ITSS
            t = [item] * ITSS
        else:
            indices = np.random.choice(len(neighbors), size=ITSS, replace=(len(neighbors) < ITSS))
            for idx in indices:
                h.append(item)
                r.append(neighbors[idx][1])
                t.append(neighbors[idx][0])
        item_triple_set[item].append((h, r, t))

print("Đang xây dựng User Triple Set (UTSS=32) từ lịch sử tương tác tích cực...")
user_history = collections.defaultdict(list)
for u, i, r in train_data:
    if r == 1:
        user_history[u].append(i)

user_triple_set = collections.defaultdict(list)
for u in range(n_user):
    liked = user_history.get(u, [0])
    entities = liked
    for l in range(N_LAYER):
        h, r, t = [], [], []
        for ent in entities:
            for tail, rel in kg_dict.get(ent, []):
                h.append(ent)
                r.append(rel)
                t.append(tail)
        if len(h) == 0:
            dummy_h = [liked[0]] * UTSS
            dummy_r = [0] * UTSS
            dummy_t = [liked[0]] * UTSS
            user_triple_set[u].append((dummy_h, dummy_r, dummy_t))
        else:
            indices = np.random.choice(len(h), size=UTSS, replace=(len(h) < UTSS))
            user_triple_set[u].append(([h[idx] for idx in indices], [r[idx] for idx in indices], [t[idx] for idx in indices]))
            entities = [t[idx] for idx in indices]

print("[OK] Khởi tạo Lan truyền Tri thức hoàn tất!")



In [ ]:
# ============================================================
# 6. ĐỊNH NGHĨA CÁC MÔ HÌNH SO SÁNH (4 MÔ HÌNH)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

# --- 6A. BASELINE 1: POPULARITY-BASED (MostPop) ---
class MostPopularRecommender:
    def __init__(self):
        self.item_counts = collections.defaultdict(int)
        self.max_count = 1.0

    def fit(self, train_data):
        for u, i, r in train_data:
            if r == 1:
                self.item_counts[i] += 1
        self.max_count = max(self.item_counts.values()) if self.item_counts else 1.0

    def predict(self, users, items):
        scores = [self.item_counts.get(i, 0) / self.max_count for i in items]
        return np.array(scores)

# --- 6B. BASELINE 2: ITEM-KNN (Collaborative Filtering) ---
class ItemKNNRecommender:
    def __init__(self, k=20):
        self.k = k
        self.user_pos = collections.defaultdict(set)
        self.item_pos = collections.defaultdict(set)

    def fit(self, train_data):
        for u, i, r in train_data:
            if r == 1:
                self.user_pos[u].add(i)
                self.item_pos[i].add(u)

    def predict_one(self, u, i):
        u_liked = self.user_pos.get(u, set())
        if not u_liked:
            return 0.5
        i_users = self.item_pos.get(i, set())
        if not i_users:
            return 0.5
        sims = []
        for liked_item in u_liked:
            liked_users = self.item_pos.get(liked_item, set())
            intersection = len(i_users & liked_users)
            denom = np.sqrt(len(i_users) * len(liked_users))
            if denom > 0:
                sims.append(intersection / denom)
        if not sims:
            return 0.5
        top_sims = sorted(sims, reverse=True)[:self.k]
        return float(np.mean(top_sims))

    def predict(self, users, items):
        return np.array([self.predict_one(u, i) for u, i in zip(users, items)])

# --- 6C. BASELINE 3: BIASED MATRIX FACTORIZATION (MF - NO KG) ---
class MatrixFactorization(nn.Module):
    def __init__(self, n_user, n_item, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_user, dim)
        self.item_emb = nn.Embedding(n_item, dim)
        self.user_bias = nn.Embedding(n_user, 1)
        self.item_bias = nn.Embedding(n_item, 1)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        u = self.user_emb(users)
        v = self.item_emb(items)
        dot = (u * v).sum(dim=-1, keepdim=True)
        logits = dot + self.user_bias(users) + self.item_bias(items)
        return torch.sigmoid(logits.squeeze(-1))

    def get_all_scores(self, user_indices):
        u_emb = self.user_emb(user_indices)
        v_emb = self.item_emb.weight
        u_bias = self.user_bias(user_indices)
        v_bias = self.item_bias.weight.T
        logits = torch.matmul(u_emb, v_emb.T) + u_bias + v_bias
        return torch.sigmoid(logits)

# --- 6D. MÔ HÌNH ĐỀ XUẤT: CKAN (WITH KNOWLEDGE GRAPH) ---
class CKAN(nn.Module):
    def __init__(self, n_entity, n_relation, dim, n_layer=1, agg="concat"):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        self.agg = agg

        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)
        self.attention = nn.Sequential(
            nn.Linear(dim * 2, dim, bias=False), nn.ReLU(),
            nn.Linear(dim, dim, bias=False), nn.ReLU(),
            nn.Linear(dim, 1, bias=False), nn.Sigmoid()
        )
        self._init_weight()

    def _init_weight(self):
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        for m in self.attention:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def _knowledge_attention(self, h_emb, r_emb, t_emb):
        alpha = self.attention(torch.cat((h_emb, r_emb), dim=-1)).squeeze(-1)
        alpha = F.softmax(alpha, dim=-1)
        return torch.mul(alpha.unsqueeze(-1), t_emb).sum(dim=1)

    def get_user_embeddings(self, user_triple):
        user_embs = [self.entity_emb(user_triple[0][0]).mean(dim=1)]
        for l in range(self.n_layer):
            h = self.entity_emb(user_triple[0][l])
            r = self.relation_emb(user_triple[1][l])
            t = self.entity_emb(user_triple[2][l])
            user_embs.append(self._knowledge_attention(h, r, t))

        e_u = user_embs[0]
        if self.agg == "concat":
            for i in range(1, len(user_embs)):
                e_u = torch.cat((user_embs[i], e_u), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(user_embs)):
                e_u = e_u + user_embs[i]
        return e_u

    def get_item_embeddings(self, items, item_triple):
        item_embs = [self.entity_emb(items)]
        for l in range(self.n_layer):
            h = self.entity_emb(item_triple[0][l])
            r = self.relation_emb(item_triple[1][l])
            t = self.entity_emb(item_triple[2][l])
            item_embs.append(self._knowledge_attention(h, r, t))

        e_v = item_embs[0]
        if self.agg == "concat":
            for i in range(1, len(item_embs)):
                e_v = torch.cat((item_embs[i], e_v), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(item_embs)):
                e_v = e_v + item_embs[i]
        return e_v

    def forward(self, items, user_triple, item_triple):
        e_u = self.get_user_embeddings(user_triple)
        e_v = self.get_item_embeddings(items, item_triple)
        return torch.sigmoid((e_v * e_u).sum(dim=-1))

print("[OK] Đã khởi tạo cấu trúc 4 mô hình so sánh!")



In [ ]:
# ============================================================
# 7. HUẤN LUYỆN & ĐÁNH GIÁ CTR BENCHMARK (Warm-start)
# ============================================================
def to_triple_tensor(objs, triple_set, n_layer, dev):
    h, r, t = [], [], []
    for i in range(n_layer):
        h.append(torch.LongTensor([triple_set[o][i][0] for o in objs]).to(dev))
        r.append(torch.LongTensor([triple_set[o][i][1] for o in objs]).to(dev))
        t.append(torch.LongTensor([triple_set[o][i][2] for o in objs]).to(dev))
    return [h, r, t]

def eval_torch_model(model, data, is_ckan=False, b_size=BATCH_SIZE):
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for start in range(0, len(data), b_size):
            batch = data[start:start+b_size]
            u = torch.LongTensor(batch[:, 0]).to(device)
            i = torch.LongTensor(batch[:, 1]).to(device)
            lbl = batch[:, 2]
            if is_ckan:
                u_tr = to_triple_tensor(batch[:, 0], user_triple_set, N_LAYER, device)
                i_tr = to_triple_tensor(batch[:, 1], item_triple_set, N_LAYER, device)
                sc = model(i, u_tr, i_tr).cpu().numpy()
            else:
                sc = model(u, i).cpu().numpy()
            scores.extend(sc)
            labels.extend(lbl)
    auc = roc_auc_score(labels, scores)
    f1 = f1_score(labels, [1 if s >= 0.5 else 0 for s in scores])
    return auc, f1

# 1. MostPopular
mostpop = MostPopularRecommender()
mostpop.fit(train_data)
pop_scores = mostpop.predict(test_data[:, 0], test_data[:, 1])
pop_auc = roc_auc_score(test_data[:, 2], pop_scores)
pop_f1 = f1_score(test_data[:, 2], [1 if s >= 0.5 else 0 for s in pop_scores])
print(f"[1/4] MostPopular       : Test AUC={pop_auc:.4f}, F1={pop_f1:.4f}")

# 2. Item-KNN
itemknn = ItemKNNRecommender(k=20)
itemknn.fit(train_data)
sub_test = test_data[:5000] # Subsample for speed
knn_scores = itemknn.predict(sub_test[:, 0], sub_test[:, 1])
knn_auc = roc_auc_score(sub_test[:, 2], knn_scores)
knn_f1 = f1_score(sub_test[:, 2], [1 if s >= 0.5 else 0 for s in knn_scores])
print(f"[2/4] Item-KNN          : Test AUC={knn_auc:.4f}, F1={knn_f1:.4f}")

# 3. Matrix Factorization
mf_model = MatrixFactorization(n_user, n_item, DIM).to(device)
mf_opt = torch.optim.Adam(mf_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
criterion = nn.BCELoss()

print("Đang huấn luyện Matrix Factorization...")
for ep in range(1, 11):
    mf_model.train()
    indices = np.random.permutation(len(train_data))
    for st in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[indices[st:st+BATCH_SIZE]]
        u = torch.LongTensor(batch[:, 0]).to(device)
        i = torch.LongTensor(batch[:, 1]).to(device)
        lbl = torch.FloatTensor(batch[:, 2]).to(device)
        mf_opt.zero_grad()
        loss = criterion(mf_model(u, i), lbl)
        loss.backward()
        mf_opt.step()
mf_auc, mf_f1 = eval_torch_model(mf_model, test_data, is_ckan=False)
print(f"[3/4] Matrix Factorization: Test AUC={mf_auc:.4f}, F1={mf_f1:.4f}")

# 4. CKAN
ckan_model = CKAN(n_entity, n_relation, DIM, N_LAYER, AGG).to(device)
ckan_opt = torch.optim.Adam(ckan_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)

print("Đang huấn luyện CKAN với Knowledge Graph...")
for ep in range(1, 11):
    ckan_model.train()
    indices = np.random.permutation(len(train_data))
    for st in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[indices[st:st+BATCH_SIZE]]
        i = torch.LongTensor(batch[:, 1]).to(device)
        lbl = torch.FloatTensor(batch[:, 2]).to(device)
        u_tr = to_triple_tensor(batch[:, 0], user_triple_set, N_LAYER, device)
        i_tr = to_triple_tensor(batch[:, 1], item_triple_set, N_LAYER, device)
        ckan_opt.zero_grad()
        loss = criterion(ckan_model(i, u_tr, i_tr), lbl)
        loss.backward()
        ckan_opt.step()
ckan_auc, ckan_f1 = eval_torch_model(ckan_model, test_data, is_ckan=True)
print(f"[4/4] CKAN (With KG)     : Test AUC={ckan_auc:.4f}, F1={ckan_f1:.4f}")



In [ ]:
# ============================================================
# 8. THỰC NGHIỆM ĐỘ THƯA THỚT DỮ LIỆU (DATA SPARSITY STUDY)
# Đòn bẩy chứng minh: Khi giảm dữ liệu, MF sụp đổ, CKAN vững vàng
# ============================================================
sparsity_ratios = [0.10, 0.20, 0.50, 1.00]
sparsity_results = {"Ratio": [], "MF_AUC": [], "CKAN_AUC": []}

print("=== BẮT ĐẦU NGHIÊN CỨU SỨC CHỊU ĐỰNG TRÊN DỮ LIỆU THƯA THỚT ===")
for r in sparsity_ratios:
    sub_size = int(len(train_data) * r)
    sub_train = train_data[:sub_size]
    print(f"\n--- Đang đánh giá tỷ lệ dữ liệu: {int(r*100)}% ({sub_size:,} tương tác) ---")

    # Huấn luyện MF nhanh 5 epochs
    m_sub = MatrixFactorization(n_user, n_item, DIM).to(device)
    opt_m = torch.optim.Adam(m_sub.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
    for _ in range(5):
        m_sub.train()
        for st in range(0, len(sub_train), BATCH_SIZE):
            b = sub_train[st:st+BATCH_SIZE]
            opt_m.zero_grad()
            l = criterion(m_sub(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)), torch.FloatTensor(b[:, 2]).to(device))
            l.backward()
            opt_m.step()
    auc_m, _ = eval_torch_model(m_sub, test_data, is_ckan=False)

    # Huấn luyện CKAN nhanh 5 epochs
    c_sub = CKAN(n_entity, n_relation, DIM, N_LAYER, AGG).to(device)
    opt_c = torch.optim.Adam(c_sub.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
    for _ in range(5):
        c_sub.train()
        for st in range(0, len(sub_train), BATCH_SIZE):
            b = sub_train[st:st+BATCH_SIZE]
            u_tr = to_triple_tensor(b[:, 0], user_triple_set, N_LAYER, device)
            i_tr = to_triple_tensor(b[:, 1], item_triple_set, N_LAYER, device)
            opt_c.zero_grad()
            l = criterion(c_sub(torch.LongTensor(b[:, 1]).to(device), u_tr, i_tr), torch.FloatTensor(b[:, 2]).to(device))
            l.backward()
            opt_c.step()
    auc_c, _ = eval_torch_model(c_sub, test_data, is_ckan=True)

    print(f"Kết quả ({int(r*100)}% Data): MF AUC = {auc_m:.4f} | CKAN AUC = {auc_c:.4f}")
    sparsity_results["Ratio"].append(f"{int(r*100)}%")
    sparsity_results["MF_AUC"].append(auc_m)
    sparsity_results["CKAN_AUC"].append(auc_c)

df_sp = pd.DataFrame(sparsity_results)
print("\nBẢNG KẾT QUẢ TỔNG HỢP SPARSITY:")
print(df_sp.to_string(index=False))



In [ ]:
# ============================================================
# 9. TRỰC QUAN HÓA BÁO CÁO TOÀN DIỆN
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

# Biểu đồ 1: So sánh tổng quan 4 mô hình
models = ["MostPopular", "Item-KNN", "Biased MF", "CKAN (KG)"]
aucs = [pop_auc, knn_auc, mf_auc, ckan_auc]
colors = ["#9CA3AF", "#60A5FA", "#3B82F6", "#D97706"]

bars = axes[0].bar(models, aucs, color=colors, edgecolor="#1F2937", linewidth=1.2)
axes[0].set_ylim(0.4, 1.05)
axes[0].set_ylabel("Test ROC-AUC", fontsize=11, fontweight="bold")
axes[0].set_title("So Sánh CTR Prediction (Warm-Start Benchmark)", fontsize=11.5, fontweight="bold")
axes[0].grid(axis="y", linestyle=":", alpha=0.7)
for b in bars:
    h = b.get_height()
    axes[0].annotate(f"{h:.4f}", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4),
                     textcoords="offset points", ha="center", va="bottom", weight="bold")

# Biểu đồ 2: Sức chịu đựng trên dữ liệu thưa thớt
axes[1].plot(sparsity_results["Ratio"], sparsity_results["CKAN_AUC"], label="CKAN (With KG)", color="#D97706", linewidth=2.8, marker="o", markersize=7)
axes[1].plot(sparsity_results["Ratio"], sparsity_results["MF_AUC"], label="Matrix Factorization", color="#3B82F6", linewidth=2.2, linestyle="--", marker="s", markersize=7)
axes[1].set_xlabel("Tỷ lệ dữ liệu huấn luyện (Training Ratio)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Test ROC-AUC", fontsize=11, fontweight="bold")
axes[1].set_title("Tác Động Của Độ Thưa Thớt: MF Sụp Đổ vs CKAN Vững Bền", fontsize=11.5, fontweight="bold", color="#991B1B")
axes[1].legend(loc="lower right")
axes[1].grid(True, linestyle=":", alpha=0.7)

plt.tight_layout()
plt.savefig("./results/colab_benchmark_comparison.png", bbox_inches="tight")
plt.show()
print("[OK] Đã hoàn thành toàn bộ thực nghiệm và lưu đồ thị!")

